# **`NumPy` para economistas · Parte 2**

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/franperezec/propedeutico-flacso/blob/main/clases/matrices/numpy_para_economistas_02.ipynb)

**Profesor:** Francisco Pérez M. · **Propedéutico de Matemáticas y Programación · FLACSO Ecuador**


# Arreglos y matrices con NumPy 2

**Correo:** francisco.perezxxi@gmail.com

---

Información sobre la librería de `Python` para trabajar con `Numpy` y `Google Sheets`:
[algebra-lineal.sheets](https://pypi.org/project/algebra-lineal-sheets/)
En clase resolvieron a mano sistemas $2\times2$ y $3\times3$, calcularon determinantes
y diagonalizaron matrices pequeñas: $A = PDP^{-1}$. Hoy hacemos exactamente lo mismo,
con una diferencia: **el tamaño deja de ser una restricción**. El precio de ese poder
son algunas trampas numéricas que el papel no tiene, y también las vamos a ver.

Una distinción atraviesa toda la sesión: un **arreglo** de NumPy es una estructura de
datos flexible; una **matriz** es un objeto algebraico con reglas propias. Un arreglo
*puede representar* una matriz, pero no cumple el álgebra matricial por sí solo — hay
que pedírsela con los operadores correctos.

## Cronograma

| B | Objetivo | Actividad central | min | acum |
|---|----------|-------------------|-----|------|
| 0 | Entorno listo y regla de comprobación | `git pull`, kernel, `comprobar()` | 8 | 8 |
| 1 | Distinguir arreglo de matriz: `shape`, orientación, `dtype` | Yo hago + **Tú haces 1** | 16 | 24 |
| 2 | Operar sin bucles: vectorización, `axis`, *broadcasting* | Yo hago + **Tú haces 2** | 18 | 42 |
| 3 | Álgebra matricial: `*` vs `@`, traza, matrices especiales | Yo hago + **Tú haces 3** | 18 | 60 |
| 4 | Determinante, rango y $Ax=b$ a cualquier escala | Yo hago + **Tú haces 4** + Leontief | 26 | 86 |
| 5 | Descomposición $A=PDP^{-1}$ y verla actuar | Comprobación guiada + Markov | 24 | 110 |
| 6 | Puente a la maestría: `lstsq`; ticket de salida; deber | Cierre | 10 | 120 |

## Metodología

Cada bloque sigue el ciclo: **Yo hago** (ejemplo comentado) → **Predicción** (anticipan
resultado y `shape` antes de ejecutar) → **Tú haces** (5–7 minutos, celda vacía con la
solución colapsada debajo: ábranla *después* de intentarlo) → **Comprobamos** (validación
numérica e interpretación).

**Regla de oro**: nada se acepta sin comprobación numérica explícita.

Este cuaderno solo requiere `numpy`. El cuaderno **ampliado** (referencia y deber)
profundiza cada bloque.

---
## 0. Puesta en marcha (8')

**En la terminal, antes de abrir el notebook** (desde la carpeta del repositorio del
curso):

```bash
git status          # debe estar limpio; si hay cambios locales, avísenme antes de continuar
git pull --ff-only  # trae el material sin crear merges
```

Flujo de dos carpetas: el repositorio del curso se mantiene limpio (solo recibe `pull`);
**copien** este `.ipynb` a su carpeta personal de trabajo y ábranlo ahí.

**En VS Code**: seleccionen el kernel de su `.venv` (esquina superior derecha). La celda
siguiente verifica que de verdad están en él: el dato clave es `sys.executable`.

In [1]:
import sys
import time
import numpy as np

print("Python :", sys.version.split()[0])
print("Kernel :", sys.executable, "   <- en VS Code debe apuntar a su .venv")
print("NumPy  :", np.__version__)
print("Entorno:", "Google Colab" if "google.colab" in sys.modules else "local (VS Code / Jupyter)")

# Salidas legibles: 4 decimales, sin notación científica para números pequeños.
np.set_printoptions(precision=4, suppress=True)

Python : 3.12.10
Kernel : C:\Users\econo\OneDrive\FLACSO\Clases\Maestría\2026Prope\.venv\Scripts\python.exe    <- en VS Code debe apuntar a su .venv
NumPy  : 2.5.1
Entorno: local (VS Code / Jupyter)


Dos reglas que arrastramos toda la sesión:

1. **Semilla fija**: cada celda con azar crea su propio generador con una semilla
   fija (la semilla puede variar entre ejercicios). Cada resultado es reproducible y
   no depende del orden de ejecución.
2. **Nunca comparar flotantes con `==`**: la aritmética de punto flotante no es exacta.
   Nuestra función `comprobar` valida con tolerancia **y además exige que las formas
   coincidan** — un `shape` equivocado es un error, aunque los valores parezcan iguales.

In [2]:
def comprobar(etiqueta, obtenido, esperado, rtol=1e-9, atol=1e-10):
    # Compara FORMA y valores. Shapes distintas cuentan como FALLA: np.allclose por sí
    # solo haría broadcasting y podría ocultar justo el tipo de error que hoy aprendemos
    # a detectar.
    obtenido = np.asarray(obtenido)
    esperado = np.asarray(esperado)
    if obtenido.shape != esperado.shape:
        print(f"[FALLA] {etiqueta:<50} shapes distintas: {obtenido.shape} vs {esperado.shape}")
        return False
    ok = np.allclose(obtenido, esperado, rtol=rtol, atol=atol)
    error = np.max(np.abs(obtenido - esperado)) if obtenido.size else 0.0
    print(f"[{'OK   ' if ok else 'FALLA'}] {etiqueta:<50} error máx = {error:.2e}")
    return ok


# Por qué existe esta función:
print("0.1 + 0.2 == 0.3           ->", 0.1 + 0.2 == 0.3)
print("diferencia real            ->", 0.1 + 0.2 - 0.3)
comprobar("0.1 + 0.2 ≈ 0.3 (con tolerancia)", 0.1 + 0.2, 0.3)

0.1 + 0.2 == 0.3           -> False
diferencia real            -> 5.551115123125783e-17
[OK   ] 0.1 + 0.2 ≈ 0.3 (con tolerancia)                   error máx = 5.55e-17


True

---
## 1. El arreglo no es una matriz (16')

Una **matriz** de álgebra lineal es un objeto algebraico: un mapa lineal, con su producto
definido como $(AB)_{ij}=\sum_k a_{ik}b_{kj}$.

Un **`ndarray`** es una estructura de datos: un arreglo homogéneo de $N$ dimensiones
descrito por su `dtype`, su `shape` y la manera en que se recorre la memoria (puede ser
un bloque propio o una *vista* sobre otro arreglo). El álgebra **no viene incluida**:
hay que invocarla con `@` y con `np.linalg`. Todo lo demás es aritmética punto a punto.

$$
\texttt{A * B} \;\to\; \text{elemento a elemento } (a_{ij}b_{ij})
\qquad\qquad
\texttt{A @ B} \;\to\; \text{producto matricial } \Big(\textstyle\sum_k a_{ik}b_{kj}\Big)
$$

(Existe una clase `np.matrix` donde `*` sí multiplica matrices: está desaconsejada por
NumPy y no la usamos.)

In [3]:
# --- Un vector 1D no es fila ni columna -------------------------------------
v = np.array([3.0, -1.0, 2.0])
print("v         =", v)
print("v.shape   =", v.shape, " | v.ndim =", v.ndim, " | v.size =", v.size, " | dtype:", v.dtype)
print("v.T.shape =", v.T.shape, "  <- transponer un 1D NO hace nada")

v         = [ 3. -1.  2.]
v.shape   = (3,)  | v.ndim = 1  | v.size = 3  | dtype: float64
v.T.shape = (3,)   <- transponer un 1D NO hace nada


**Predicción** (antes de ejecutar): si `fila` tiene forma $(1,3)$ y `columna` forma
$(3,1)$, ¿qué `shape` tiene `fila @ columna`? ¿Y `columna @ fila`? ¿Alguno es un escalar?

In [4]:
%pip install -q "algebra-lineal-sheets>=1.4.0"

Note: you may need to restart the kernel to use updated packages.


In [5]:
import algebra_lineal as al
# Importar
from algebra_lineal import *  # traer librerías
version()  # versión de librerías
import numpy as np
import matplotlib.pyplot as plt


📚 ALGEBRA LINEAL v1.4.0
🎓 Para estudiantes de álgebra lineal
🔧 Ejecuta: configurar() para empezar
📖 Ayuda completa: ayuda()
🔗 PyPI: https://pypi.org/project/algebra-lineal-sheets/
📦 ALGEBRA LINEAL v1.4.0
👨‍🏫 Autor: Francisco Pérez Mogollón
📚 Álgebra lineal simplificada con Google Sheets y Excel
🔗 PyPI: https://pypi.org/project/algebra-lineal-sheets/
🛠️  Instalación: pip install algebra-lineal-sheets
🛠️  Con Google Sheets: pip install "algebra-lineal-sheets[google]"


In [6]:
# MODO EXCEL LOCAL: las matrices que exportemos van a un Excel de resultados
configurar(sheet='resultados.xlsx')   # el archivo se crea con el primer exportar()

# MODO GOOGLE SHEETS (en Colab, como en la clase):
#configurar()                          # busca el Google Sheet llamado 'matrices'

📊 Modo Excel local: resultados.xlsx
⚠️  El archivo aún no existe: exportar() lo creará
📖 Usa ayuda() para ver todos los comandos disponibles


True

In [7]:
# cambiar_sheet() cambia el destino activo en cualquier momento:
#cambiar_sheet('prope2026')            # a un Google Sheet (modo Colab)
#cambiar_sheet('otro_archivo.xlsx')    # a otro Excel local

In [8]:
columna = v.reshape(-1, 1)      # (3,1): el -1 significa "infiere esta dimensión"
fila    = v.reshape(1, -1)      # (1,3)
print("columna:", columna.shape, "| fila:", fila.shape)

print("\nfila @ columna =", fila @ columna, " shape:", (fila @ columna).shape,
      " <- MATRIZ 1×1 (su único elemento es el producto interno)")
print("columna @ fila shape:", (columna @ fila).shape, " <- matriz 3×3 (producto externo)")

# Solo el producto de dos arreglos 1D devuelve un escalar de verdad:
print("\nv @ v =", v @ v, " | tipo:", type(v @ v).__name__, " <- escalar, sin envoltura")
print("\nLección: el significado y la forma del resultado dependen de las formas de entrada.")

columna: (3, 1) | fila: (1, 3)

fila @ columna = [[14.]]  shape: (1, 1)  <- MATRIZ 1×1 (su único elemento es el producto interno)
columna @ fila shape: (3, 3)  <- matriz 3×3 (producto externo)

v @ v = 14.0  | tipo: float64  <- escalar, sin envoltura

Lección: el significado y la forma del resultado dependen de las formas de entrada.


In [9]:
exportar('fila', 'columna')

✅ fila → creada
✅ columna → creada
🎯 ¡Exportadas exitosamente!: fila, columna
🔗 Revisa tu archivo — 📊 Excel local: resultados.xlsx


['fila', 'columna']

In [10]:
# --- El dtype no es cosmético ------------------------------------------------
enteros = np.array([[1, 2], [3, 4]])
print("dtype:", enteros.dtype)
print("enteros / 2 =\n", enteros / 2, " <- la división promueve a float")
print("dtype de su inversa:", np.linalg.inv(enteros).dtype, " <- np.linalg trabaja en float")
print("Regla: para álgebra lineal, declaren float desde el inicio (1.0 o dtype=float).")

# --- reshape en 10 segundos --------------------------------------------------
u = np.arange(1.0, 13.0)
M = u.reshape(3, 4)              # se llena POR FILAS; size se conserva: 12 = 3·4
print("\nreshape(3,4):\n", M)
print("de vuelta a 1D:", M.reshape(-1))
print("(reshape devuelve una vista cuando la memoria lo permite y una copia cuando no;")
print(" los detalles, y los arreglos 3D, están en el cuaderno ampliado.)")

dtype: int64
enteros / 2 =
 [[0.5 1. ]
 [1.5 2. ]]  <- la división promueve a float
dtype de su inversa: float64  <- np.linalg trabaja en float
Regla: para álgebra lineal, declaren float desde el inicio (1.0 o dtype=float).

reshape(3,4):
 [[ 1.  2.  3.  4.]
 [ 5.  6.  7.  8.]
 [ 9. 10. 11. 12.]]
de vuelta a 1D: [ 1.  2.  3.  4.  5.  6.  7.  8.  9. 10. 11. 12.]
(reshape devuelve una vista cuando la memoria lo permite y una copia cuando no;
 los detalles, y los arreglos 3D, están en el cuaderno ampliado.)


### Tú haces 1 (5')

Con `z = np.arange(5.0)`:

1. Reporten `shape` y `ndim` de `z`.
2. Construyan `col` de forma $(5,1)$ y `fil` de forma $(1,5)$.
3. Calculen $z^\top z$ de dos maneras: `z @ z` y `fil @ col`. Reporten el **valor** y el
   **shape** de cada resultado. ¿Cuál es un escalar y cuál una matriz $1\times1$?
4. Extraigan el número de la matriz $1\times1$ con `.item()` y verifiquen con
   `comprobar` que coincide con `z @ z`.

In [11]:
# Tu código aquí

<details>
<summary><b>▶ Ver solución</b></summary>

```python
z = np.arange(5.0)
print("z:", z, "| shape:", z.shape, "| ndim:", z.ndim)

col = z.reshape(-1, 1)
fil = z.reshape(1, -1)
print("col:", col.shape, "| fil:", fil.shape)

a = z @ z                        # escalar
B = fil @ col                    # matriz (1,1)
print("\nz @ z     =", a, "        | shape:", np.shape(a), " <- escalar")
print("fil @ col =", B, " | shape:", B.shape, " <- matriz 1×1")

comprobar("mismo valor, distinta envoltura", B.item(), a)
print("\nMoraleja: el valor coincide; la FORMA del resultado la dictan las formas de entrada.")
```

</details>

## 2. Operar sin bucles: vectorización, `axis` y *broadcasting* (18')

Aquí empieza la respuesta a *"¿para qué el código si ya sé hacerlo a mano?"*.

A diferencia de un bucle `for` de Python, que paga el costo de ser evaluado por el intérprete en cada iteración, NumPy aplica **vectorización**. Esto significa delegar el cálculo a niveles más profundos y eficientes del hardware:

1. **Operaciones elemento a elemento (aritmética básica):** Se ejecutan mediante bucles precompilados en C. Además, NumPy aprovecha instrucciones **SIMD** (*Single Instruction, Multiple Data*). En lugar de sumar elementos uno por uno ($a_1 + b_1$, luego $a_2 + b_2$), el procesador ejecuta una única instrucción que opera sobre bloques enteros de datos en el mismo ciclo de reloj.
2. **Álgebra lineal pesada (producto de matrices, determinantes):** Se delega a **BLAS** y **LAPACK**. Estas son librerías industriales de bajo nivel (escritas en C y Fortran) que representan el estándar de oro en cálculo científico. Están hiperoptimizadas para gestionar la memoria de la computadora de forma inteligente al realizar operaciones matriciales.

La ventaja de evitar el intérprete de Python es **sustancial**: el código vectorizado suele ser de uno a tres órdenes de magnitud más rápido ($10\times$ a $1000\times$). El factor exacto variará según la complejidad matemática, el tamaño de la matriz (como veremos al escalar a $800 \times 800$) y la arquitectura de su procesador.

---

### Desglose técnico:

* **SIMD:** Es paralelismo puro a nivel de CPU. Si tienes registros de 256 bits, puedes empaquetar cuatro flotantes de 64 bits (`float64`) y sumarlos con otros cuatro en una sola operación de CPU. Esto liquida por completo el enfoque secuencial de un `for`.
* **BLAS (*Basic Linear Algebra Subprograms*):** Son rutinas de nivel 1, 2 y 3. Nivel 1 opera vectores, Nivel 2 opera matriz-vector, Nivel 3 opera matriz-matriz (donde la optimización de caché del procesador brilla).
* **LAPACK (*Linear Algebra PACKage*):** Se monta sobre BLAS para resolver sistemas complejos: factorizaciones (LU, Cholesky, QR) y la descomposición espectral/eigenvalores que tienes en el bloque 5 del cuaderno. Las distribuciones de NumPy en Colab suelen usar implementaciones como *OpenBLAS* o *Intel MKL* bajo el capó.

In [12]:
rng = np.random.default_rng(42)
n = 200_000
x = rng.normal(size=n)
y = rng.normal(size=n)

t0 = time.perf_counter()
z_bucle = [x[i] * y[i] for i in range(n)]
t_bucle = time.perf_counter() - t0

t0 = time.perf_counter()
z_vect = x * y
t_vect = time.perf_counter() - t0

print(f"bucle Python : {t_bucle*1000:8.2f} ms")
print(f"vectorizado  : {t_vect*1000:8.2f} ms   (~{t_bucle/max(t_vect,1e-9):,.0f}× en esta máquina)")
print("(La ventaja suele ser sustancial; el factor y el orden de magnitud varían")
print(" con la operación, el tamaño, el hardware y la implementación.)")

bucle Python :    43.84 ms
vectorizado  :     0.60 ms   (~73× en esta máquina)
(La ventaja suele ser sustancial; el factor y el orden de magnitud varían
 con la operación, el tamaño, el hardware y la implementación.)


### `axis`: el eje que se indica es el eje que **desaparece**

Una matriz de datos en economía es $n\times k$: $n$ observaciones en filas, $k$
variables en columnas. Entonces `axis=0` colapsa las filas y devuelve un estadístico
**por variable**; `axis=1` colapsa columnas y devuelve uno **por observación**.

### *Broadcasting*: operar formas distintas sin replicar datos

NumPy alinea las formas **por la derecha** y en cada posición: si son iguales, avanza;
si una es $1$, la estira; si faltan dimensiones a la izquierda, las agrega como $1$; en
cualquier otro caso, error.

$$
\underbrace{(5,3)}_{X} + \underbrace{(3,)}_{\text{medias por columna}} \to (5,3)\ \checkmark
\qquad\qquad
\underbrace{(5,3)}_{X} + \underbrace{(5,)}_{\text{medias por fila}} \to \text{error}
$$

El segundo caso alinea $3$ con $5$ y falla. La cura es `keepdims=True`, que conserva el
eje colapsado como dimensión de tamaño $1$: la media por fila sale con forma $(5,1)$ y
entonces sí se estira.

In [13]:
# 1. Generación de datos: Semilla moderna para reproducibilidad
rng = np.random.default_rng(42)

# Construimos una matriz X de 5 observaciones y 3 variables.
# NumPy aplica broadcasting internamente: asigna el primer valor de 'loc' y 'scale'
# a la primera columna, el segundo a la segunda, y así sucesivamente.
X = rng.normal(
    loc=[10.0, 100.0, 0.5],  # μ: Medias de las variables 1, 2 y 3
    scale=[2.0, 15.0, 0.1],  # σ: Desviaciones estándar de las variables 1, 2 y 3
    size=(5, 3)              # 5 filas (observaciones) × 3 columnas (variables)
)
print("X (5 observaciones × 3 variables):\n", X)


# 2. Agregaciones: Regla mnemotécnica -> "el axis que indicas es el que colapsa"

# axis=0: Colapsa el eje de las filas. La operación recorre verticalmente.
# Resultado: Vector 1D de tamaño (3,). Obtienes una métrica por VARIABLE.
print("\n--- Operaciones por Variable (axis=0) ---")
print("Media:", X.mean(axis=0))
print("Shape:", X.mean(axis=0).shape)

# axis=1: Colapsa el eje de las columnas. La operación recorre horizontalmente.
# Resultado: Vector 1D de tamaño (5,). Obtienes una métrica por OBSERVACIÓN.
print("\n--- Operaciones por Observación (axis=1) ---")
print("Media:", X.mean(axis=1))
print("Shape:", X.mean(axis=1).shape)

# CUIDADO: Por defecto, NumPy usa ddof=0 (desviación poblacional, divide por n).
# En estadística y econometría trabajamos con muestras, por lo que forzamos ddof=1 (n-1).
print("\nDesviación estándar muestral (axis=0, ddof=1) =", X.std(axis=0, ddof=1))


# 3. Broadcasting y Prevención de Errores: keepdims
# Al colapsar con axis=1, obtenemos un shape (5,). Si intentamos restar esto de X (5, 3),
# NumPy alineará a la derecha y fallará al chocar un 3 contra un 5.
# keepdims=True retiene el eje colapsado como una dimensión de tamaño 1 -> shape (5, 1).
# Esto permite que el broadcasting "estire" la columna para operar sobre toda la matriz.
print("\n--- keepdims en acción ---")
media_observaciones_segura = X.mean(axis=1, keepdims=True)
print("Media por fila con keepdims -> shape:", media_observaciones_segura.shape)

X (5 observaciones × 3 variables):
 [[ 10.6094  84.4002   0.575 ]
 [ 11.8811  70.7345   0.3698]
 [ 10.2557  95.2564   0.4983]
 [  8.2939 113.191    0.5778]
 [ 10.1321 116.9086   0.5468]]

--- Operaciones por Variable (axis=0) ---
Media: [10.2344 96.0981  0.5135]
Shape: (3,)

--- Operaciones por Observación (axis=1) ---
Media: [31.8616 27.6618 35.3368 40.6876 42.5291]
Shape: (5,)

Desviación estándar muestral (axis=0, ddof=1) = [ 1.2873 19.4043  0.0865]

--- keepdims en acción ---
Media por fila con keepdims -> shape: (5, 1)


In [14]:
exportar('X')

✅ X → creada
🎯 ¡Exportadas exitosamente!: X
🔗 Revisa tu archivo — 📊 Excel local: resultados.xlsx


['X']

In [15]:
# Aplicación canónica: centrar y estandarizar (el paso previo a covarianzas y regresión).
medias = X.mean(axis=0)                       # (3,)
desv   = X.std(axis=0, ddof=1)                # (3,)
Z = (X - medias) / desv                       # (5,3) - (3,) -> broadcasting por columna

comprobar("medias de Z ≈ 0", Z.mean(axis=0), np.zeros(3))
comprobar("desv. de Z ≈ 1", Z.std(axis=0, ddof=1), np.ones(3))

# Y el error clásico, en vivo:
try:
    X - X.mean(axis=1)                        # (5,3) - (5,): alinea 3 con 5
except ValueError as e:
    print("\n(5,3) - (5,) falla:", e)
print("cura: X - X.mean(axis=1, keepdims=True)  -> shape",
      (X - X.mean(axis=1, keepdims=True)).shape)

[OK   ] medias de Z ≈ 0                                    error máx = 4.88e-16
[OK   ] desv. de Z ≈ 1                                     error máx = 1.11e-16

(5,3) - (5,) falla: operands could not be broadcast together with shapes (5,3) (5,) 
cura: X - X.mean(axis=1, keepdims=True)  -> shape (5, 3)


In [16]:
exportar('medias', 'desv','Z')

✅ medias → creada
✅ desv → creada
✅ Z → creada
🎯 ¡Exportadas exitosamente!: medias, desv, Z
🔗 Revisa tu archivo — 📊 Excel local: resultados.xlsx


['medias', 'desv', 'Z']

### Tú haces 2 (6')

La matriz `notas` tiene 4 estudiantes (filas) × 3 pruebas (columnas):

```python
notas = np.array([[6.5, 7.0, 5.5],
                  [8.0, 6.0, 7.5],
                  [4.5, 5.0, 6.0],
                  [9.0, 8.5, 8.0]])
```

1. Calculen la nota media **por prueba** y el promedio **por estudiante**, indicando el
   `axis` de cada caso.
2. Estandaricen por prueba: $z_{ij} = (x_{ij}-\bar x_j)/s_j$ con `ddof=1`.
3. Comprueben con `comprobar` que las medias por prueba de $Z$ son $\approx 0$ y las
   desviaciones $\approx 1$.

In [17]:
notas = np.array([[6.5, 7.0, 5.5],
                  [8.0, 6.0, 7.5],
                  [4.5, 5.0, 6.0],
                  [9.0, 8.5, 8.0]])
# Tu código aquí

<details>
<summary><b>▶ Ver solución</b></summary>

```python
notas = np.array([[6.5, 7.0, 5.5],
                  [8.0, 6.0, 7.5],
                  [4.5, 5.0, 6.0],
                  [9.0, 8.5, 8.0]])

# 1. Por prueba: colapsan los estudiantes (filas) -> axis=0.
#    Por estudiante: colapsan las pruebas (columnas) -> axis=1.
print("media por prueba     (axis=0):", notas.mean(axis=0))
print("promedio por estudiante (axis=1):", notas.mean(axis=1))

# 2. Estandarización por prueba: broadcasting de (3,) sobre (4,3).
Z = (notas - notas.mean(axis=0)) / notas.std(axis=0, ddof=1)
print("\nZ =\n", Z)

# 3.
comprobar("medias por prueba de Z ≈ 0", Z.mean(axis=0), np.zeros(3))
comprobar("desviaciones de Z ≈ 1",      Z.std(axis=0, ddof=1), np.ones(3))
```

</details>

---
## 3. Álgebra matricial y las matrices que ya conocen (18')

### 3.1 `*` no es `@`

$$
(A \odot B)_{ij} = a_{ij}b_{ij} \;\to\; \texttt{A * B}
\qquad\qquad
(AB)_{ij} = \sum_k a_{ik}b_{kj} \;\to\; \texttt{A @ B}
$$

Y el producto matricial **no conmuta**: $AB\ne BA$ en general. La misma trampa en
versión potencia: `A ** 2` eleva cada elemento; $A^2 = AA$ es
`np.linalg.matrix_power(A, 2)`.

**Predicción**: $B=\begin{pmatrix}0&1\\1&0\end{pmatrix}$ es una permutación. ¿Qué le
hace `A @ B` a la matriz $A$? ¿Y `B @ A`? (Pista: una intercambia columnas, la otra
filas. ¿Cuál es cuál?)

In [18]:
A = np.array([[1.0, 2.0],
              [3.0, 4.0]])
B = np.array([[0.0, 1.0],
              [1.0, 0.0]])

print("A * B (Hadamard):\n", A * B)
print("\nA @ B:\n", A @ B, "  <- intercambió las COLUMNAS de A")
print("\nB @ A:\n", B @ A, "  <- intercambió las FILAS de A")
print("\n¿A @ B == B @ A?", np.allclose(A @ B, B @ A), " -> no conmuta")

print("\nA ** 2 (elemento a elemento):\n", A ** 2)
comprobar("matrix_power(A,2) = A @ A", np.linalg.matrix_power(A, 2), A @ A)

# Transpuesta y traza, con sus propiedades verificadas:
rng = np.random.default_rng(42)
P_, Q_ = rng.normal(size=(4, 4)), rng.normal(size=(4, 4))
comprobar("(PQ)ᵀ = Qᵀ Pᵀ",   (P_ @ Q_).T,       Q_.T @ P_.T)
comprobar("tr(PQ) = tr(QP)", np.trace(P_ @ Q_), np.trace(Q_ @ P_))

A * B (Hadamard):
 [[0. 2.]
 [3. 0.]]

A @ B:
 [[2. 1.]
 [4. 3.]]   <- intercambió las COLUMNAS de A

B @ A:
 [[3. 4.]
 [1. 2.]]   <- intercambió las FILAS de A

¿A @ B == B @ A? False  -> no conmuta

A ** 2 (elemento a elemento):
 [[ 1.  4.]
 [ 9. 16.]]
[OK   ] matrix_power(A,2) = A @ A                          error máx = 0.00e+00
[OK   ] (PQ)ᵀ = Qᵀ Pᵀ                                      error máx = 0.00e+00
[OK   ] tr(PQ) = tr(QP)                                    error máx = 0.00e+00


True

### 3.2 Los tipos de matrices del curso, ahora construibles

| Tipo (teoría) | Construcción | Nota |
|---|---|---|
| Identidad $I_n$ | `np.eye(n)` | neutro de `@` |
| Escalar $kI_n$ | `k * np.eye(n)` | multiplicar por ella $=$ multiplicar por $k$ |
| Diagonal | `np.diag([d1, ..., dn])` | `np.diag(A)` **extrae** la diagonal |
| Triangular | `np.triu(A)`, `np.tril(A)` | su determinante es el producto de la diagonal |
| Simétrica | `A + A.T` o `X.T @ X` | `X.T @ X` es la matriz de **Gram** (productos cruzados) |
| Aleatoria reproducible | `rng.normal(size=(m, n))` | siempre con semilla |

Precisión importante: `X.T @ X` **no** es la matriz de covarianzas. La covarianza
muestral exige **centrar** y dividir por $n-1$:
$S=\tfrac{1}{n-1}(X-\bar X)^\top(X-\bar X)$, que es `np.cov(X, rowvar=False)`.

In [19]:
n = 4
I  = np.eye(n)
Ek = 3.0 * np.eye(n)                       # matriz ESCALAR
Dg = np.diag([1.0, 2.0, 3.0, 4.0])         # diagonal
G  = np.arange(1.0, 17.0).reshape(4, 4)
U, L = np.triu(G), np.tril(G)               # triangulares
S  = G + G.T                                # simétrica por construcción

print("escalar 3I:\n", Ek)
print("\ndiag extrae:", np.diag(Dg), " | triu:\n", U)

comprobar("S es simétrica: S = Sᵀ", S, S.T)
comprobar("la matriz escalar ES multiplicar por el escalar: (3I)S = 3S", Ek @ S, 3.0 * S)
comprobar("I es neutro: I S = S", I @ S, S)

# Gram vs covarianza (la distinción que reaparecerá en econometría):
rng = np.random.default_rng(42)
X = rng.normal(size=(6, 3))
Xc = X - X.mean(axis=0)
comprobar("cov = Gram de X CENTRADA / (n-1)", Xc.T @ Xc / (X.shape[0] - 1),
          np.cov(X, rowvar=False))

escalar 3I:
 [[3. 0. 0. 0.]
 [0. 3. 0. 0.]
 [0. 0. 3. 0.]
 [0. 0. 0. 3.]]

diag extrae: [1. 2. 3. 4.]  | triu:
 [[ 1.  2.  3.  4.]
 [ 0.  6.  7.  8.]
 [ 0.  0. 11. 12.]
 [ 0.  0.  0. 16.]]
[OK   ] S es simétrica: S = Sᵀ                             error máx = 0.00e+00
[OK   ] la matriz escalar ES multiplicar por el escalar: (3I)S = 3S error máx = 0.00e+00
[OK   ] I es neutro: I S = S                               error máx = 0.00e+00
[OK   ] cov = Gram de X CENTRADA / (n-1)                   error máx = 1.11e-16


True

In [20]:
# --- El punto de la sesión: la escala ----------------------------------------
# A mano, multiplicar dos 3×3 son 27 multiplicaciones. Dos 800×800 son 800³ ≈ 5.1×10⁸.
m = 800
rng = np.random.default_rng(42)
Am = rng.normal(size=(m, m))
Bm = rng.normal(size=(m, m))

t0 = time.perf_counter()
Cm = Am @ Bm
t = time.perf_counter() - t0

print(f"producto {m}×{m}: {t*1000:.0f} ms para ~{m**3:,.0f} multiplicaciones-suma")
comprobar("tr(AB) = tr(BA) también a esta escala",
          np.trace(Am @ Bm), np.trace(Bm @ Am), atol=1e-6)
print("(El tiempo exacto depende de su máquina; la viabilidad, no.)")

producto 800×800: 8 ms para ~512,000,000 multiplicaciones-suma


[OK   ] tr(AB) = tr(BA) también a esta escala              error máx = 4.55e-13
(El tiempo exacto depende de su máquina; la viabilidad, no.)


In [21]:
exportar('Cm')

✅ Cm → creada


🎯 ¡Exportadas exitosamente!: Cm
🔗 Revisa tu archivo — 📊 Excel local: resultados.xlsx


['Cm']

### Tú haces 3 (6')

1. Construyan $D=\operatorname{diag}(2,5,7)$ y comprueben que
   $DD=\operatorname{diag}(4,25,49)$: las diagonales se multiplican **entrada a
   entrada**, como vieron en teoría.
2. Con `g = np.random.default_rng(7).normal(size=(3,3))`, construyan la simétrica
   $S=g+g^\top$ y comprueben $S=S^\top$ y $\operatorname{tr}(DS)=\operatorname{tr}(SD)$.
3. Con $U=\operatorname{triu}(g)$, comprueben que
   $\det(U)=\prod_i u_{ii}$ (`np.prod(np.diag(U))`): el determinante de una triangular
   es el producto de su diagonal.

In [22]:
# Tu código aquí

<details>
<summary><b>▶ Ver solución</b></summary>

```python
# 1. Diagonal por diagonal = diagonal de productos entrada a entrada.
D = np.diag([2.0, 5.0, 7.0])
comprobar("D @ D = diag(4, 25, 49)", D @ D, np.diag([4.0, 25.0, 49.0]))

# 2. Simétrica y conmutación bajo la traza.
g = np.random.default_rng(7).normal(size=(3, 3))
S = g + g.T
comprobar("S = Sᵀ", S, S.T)
comprobar("tr(DS) = tr(SD)", np.trace(D @ S), np.trace(S @ D))

# 3. Determinante de una triangular.
U = np.triu(g)
comprobar("det(U) = producto de la diagonal", np.linalg.det(U), np.prod(np.diag(U)))
print("\nLas matrices especiales no son solo nombres: heredan propiedades computables.")
```

</details>

---
## 4. Determinante, independencia lineal y $Ax=b$ a cualquier escala (26')

### 4.1 El determinante que calcularon a mano, y su límite numérico

En teoría vieron que, para una matriz **cuadrada**, $\det(A)\ne 0$ equivale a columnas
linealmente independientes (rango completo). NumPy calcula el mismo número que ustedes:
verifiquémoslo con ejemplos de pizarra. Pero en punto flotante la *prueba* del
determinante **no sirve**: un determinante que vale exactamente $0$ en teoría sale como
$\sim 10^{-16}$, y `det == 0` no tiene respuesta estable. La herramienta correcta es el
**rango numérico**, `np.linalg.matrix_rank` (calculado vía valores singulares con una
tolerancia explícita); para rectangulares, además, es la única herramienta general.

In [23]:
# El determinante coincide con el cálculo de pizarra:
A2 = np.array([[2.0, 1.0],
               [3.0, 4.0]])                 # a mano: 2·4 - 1·3 = 5
A3 = np.array([[1.0, 2.0, 0.0],
               [0.0, 3.0, 1.0],
               [2.0, 0.0, 1.0]])            # a mano (cofactores): 7
comprobar("det 2×2 de pizarra", np.linalg.det(A2), 5.0)
comprobar("det 3×3 de pizarra", np.linalg.det(A3), 7.0)

# La trampa: E tiene la columna 3 = columna 1 + columna 2 -> singular en teoría.
E = np.array([[1.2, 2.5, 3.7],
              [0.8, 1.1, 1.9],
              [2.4, 3.3, 5.7]])
print("\n¿col3 = col1 + col2?", np.allclose(E[:, 2], E[:, 0] + E[:, 1]))
print("det(E)          =", np.linalg.det(E), "  <- NO es exactamente 0")
print("¿det(E) == 0?   ", np.linalg.det(E) == 0.0, "  <- la prueba binaria falla")
print("matrix_rank(E)  =", np.linalg.matrix_rank(E), "de 3  -> columnas DEPENDIENTES")

# Contraste con una matriz genuinamente de rango completo y bien condicionada:
F = np.array([[2.0, 1.0, 0.0],
              [1.0, 3.0, 1.0],
              [0.0, 1.0, 2.0]])
print("\nmatrix_rank(F) =", np.linalg.matrix_rank(F),
      "| det(F) =", round(np.linalg.det(F), 4),
      "| cond(F) =", round(np.linalg.cond(F), 4))

[OK   ] det 2×2 de pizarra                                 error máx = 8.88e-16
[OK   ] det 3×3 de pizarra                                 error máx = 1.78e-15

¿col3 = col1 + col2? True
det(E)          = -9.05941988094127e-16   <- NO es exactamente 0
¿det(E) == 0?    False   <- la prueba binaria falla
matrix_rank(E)  = 2 de 3  -> columnas DEPENDIENTES

matrix_rank(F) = 3 | det(F) = 8.0 | cond(F) = 4.0


### 4.2 El número de condición y la regla de `solve`

$\kappa(A)=\sigma_{\max}/\sigma_{\min}$ mide cuánto amplifica $A$ los errores relativos
al resolver $Ax=b$. Regla de bolsillo: `float64` da $\sim16$ dígitos y
$\log_{10}\kappa$ estima **cuántos se pierden**. (Ojo: $\kappa$ depende de las unidades
de medida de las columnas; volveremos sobre eso en econometría.)

Y la regla de oficio: la solución de $Ax=b$ se **escribe** $x=A^{-1}b$, pero se
**calcula** con `np.linalg.solve(A, b)` (factorización LU con pivoteo — la eliminación
gaussiana que hicieron a mano, industrializada). Nunca `inv(A) @ b`: construir la
inversa cuesta más trabajo y agrega una ronda extra de error. `inv` se reserva para
cuando la inversa **misma** es el objeto de interés — en Leontief lo será.

In [24]:
# El 2×2 que resolverían por sustitución:  3x + y = 9 ;  x + 2y = 8
A = np.array([[3.0, 1.0],
              [1.0, 2.0]])
b = np.array([9.0, 8.0])
x = np.linalg.solve(A, b)
print("x =", x, " <- (2, 3), como a mano")
comprobar("A x = b", A @ x, b)

# El MISMO comando, 2000 incógnitas — el papel se rinde aquí:
n = 2000
rng = np.random.default_rng(42)
An = rng.normal(size=(n, n)) + n * np.eye(n)     # bien condicionada a propósito
bn = rng.normal(size=n)

t0 = time.perf_counter()
xn = np.linalg.solve(An, bn)
t = time.perf_counter() - t0

print(f"\nsolve {n}×{n}: {t*1000:.0f} ms  |  ||Ax - b||₂ = {np.linalg.norm(An @ xn - bn):.2e}")
print(f"cond(A) = {np.linalg.cond(An):.2f}  -> κ pequeño: solución confiable")
print("(El cuaderno ampliado muestra qué pasa cuando κ es enorme: matriz de Hilbert.)")

x = [2. 3.]  <- (2, 3), como a mano
[OK   ] A x = b                                            error máx = 0.00e+00



solve 2000×2000: 614 ms  |  ||Ax - b||₂ = 6.07e-14


cond(A) = 1.06  -> κ pequeño: solución confiable
(El cuaderno ampliado muestra qué pasa cuando κ es enorme: matriz de Hilbert.)


### 4.3 Aplicación: el modelo abierto de Leontief

Con $A$ la matriz de coeficientes técnicos y $d$ la demanda final:

$$
x = Ax + d \iff (I-A)x = d \iff x = (I-A)^{-1}d .
$$

El criterio **central** de que la economía sea productiva —que $(I-A)^{-1}$ exista y
sea no negativa— es el **radio espectral**: $\rho(A)<1$. (Que las sumas por columna
sean $<1$ es una condición *suficiente* bajo esta convención, no el criterio general.)
Noten el guiño: la productividad de una economía se decide en sus **valores propios** —
el tema del bloque siguiente.

Aquí la inversa de Leontief $(I-A)^{-1}$ sí interesa como objeto, porque sus entradas
son los **multiplicadores**. Pero para obtener $x$ dado un $d$ concreto: `solve`.

In [25]:
# Tres sectores: primario, manufactura, servicios.
A_tec = np.array([[0.20, 0.30, 0.10],
                  [0.10, 0.20, 0.30],
                  [0.20, 0.10, 0.20]])
d  = np.array([100.0, 80.0, 60.0])
I3 = np.eye(3)

rho = np.max(np.abs(np.linalg.eigvals(A_tec)))
print(f"radio espectral ρ(A) = {rho:.4f}  <  1  -> economía productiva (criterio central)")
print("sumas por columna:", A_tec.sum(axis=0), " <- < 1: condición suficiente aquí")

x = np.linalg.solve(I3 - A_tec, d)
print("\nproducción bruta requerida x =", x)
comprobar("verificación x = A x + d", x, A_tec @ x + d)

L = np.linalg.inv(I3 - A_tec)         # uso LEGÍTIMO de inv: sus entradas se interpretan
print("\ninversa de Leontief (I-A)⁻¹:\n", L)
print("multiplicadores totales (sumas de columna):", L.sum(axis=0))

radio espectral ρ(A) = 0.5637  <  1  -> economía productiva (criterio central)
sumas por columna: [0.5 0.6 0.6]  <- < 1: condición suficiente aquí

producción bruta requerida x = [212.5874 183.2168 151.049 ]
[OK   ] verificación x = A x + d                           error máx = 2.84e-14

inversa de Leontief (I-A)⁻¹:
 [[1.4219 0.5828 0.3963]
 [0.3263 1.4452 0.5828]
 [0.3963 0.3263 1.4219]]
multiplicadores totales (sumas de columna): [2.1445 2.3543 2.4009]


### Tú haces 4 (7')

**(a)** Diagnostiquen y resuelvan el sistema $Mx=b$ con

$$
M=\begin{pmatrix}2&1&0\\1&3&1\\0&1&2\end{pmatrix},\qquad
b=\begin{pmatrix}5\\10\\7\end{pmatrix}.
$$

Reporten $\det(M)$, $\operatorname{rango}(M)$ y $\kappa(M)$; resuelvan con `solve` y
comprueben el residuo $\lVert Mx-b\rVert_2$.

**(b)** Mismo flujo, otra escala: generen con `rng = np.random.default_rng(9)` un
sistema de $500\times500$ bien condicionado
(`A = rng.normal(size=(500,500)) + 500*np.eye(500)`, `b = rng.normal(size=500)`),
resuélvanlo y reporten el residuo. ¿Cambió algo en el *procedimiento*?

In [26]:
# Tu código aquí

<details>
<summary><b>▶ Ver solución</b></summary>

```python
# (a) Diagnóstico -> solución -> verificación.
M = np.array([[2.0, 1.0, 0.0],
              [1.0, 3.0, 1.0],
              [0.0, 1.0, 2.0]])
b = np.array([5.0, 10.0, 7.0])

print("det(M)   =", round(np.linalg.det(M), 4), " (≠ 0: cuadrada -> rango completo)")
print("rango(M) =", np.linalg.matrix_rank(M), "de 3")
print("cond(M)  =", round(np.linalg.cond(M), 4), " -> muy bien condicionada")

x = np.linalg.solve(M, b)
print("\nx =", x)
comprobar("M x = b", M @ x, b)
print(f"||Mx - b||₂ = {np.linalg.norm(M @ x - b):.2e}")

# (b) Exactamente el mismo procedimiento, 500 incógnitas.
rng = np.random.default_rng(9)
n = 500
A = rng.normal(size=(n, n)) + n * np.eye(n)
b2 = rng.normal(size=n)

x2 = np.linalg.solve(A, b2)
print(f"\n500×500: ||Ax - b||₂ = {np.linalg.norm(A @ x2 - b2):.2e}",
      f"| cond(A) = {np.linalg.cond(A):.2f}")
print("Nada cambió en el procedimiento: diagnóstico, solve, residuo. Solo cambió n.")
```

</details>

---
## 5. Descomposición espectral: $A = PDP^{-1}$ en código (24')

### 5.0 Preparación en dos minutos: vectores unitarios

La norma euclidiana es $\lVert u\rVert=\sqrt{u^\top u}$ (`np.linalg.norm`) y el vector
**unitario** es $\hat u = u/\lVert u\rVert$. Lo necesitamos ya mismo, porque un vector
propio no tiene escala natural: si $Av=\lambda v$, también $A(cv)=\lambda(cv)$. NumPy
resuelve esa ambigüedad devolviendo el representante **de norma 1** — y aun así el
**signo** queda arbitrario ($\hat v$ y $-\hat v$ son ambos válidos).

In [27]:
u = np.array([3.0, 4.0])
u_hat = u / np.linalg.norm(u)
print("||u|| =", np.linalg.norm(u), " | û =", u_hat)
comprobar("||û|| = 1", np.linalg.norm(u_hat), 1.0)

||u|| = 5.0  | û = [0.6 0.8]
[OK   ] ||û|| = 1                                          error máx = 0.00e+00


True

### 5.1 `np.linalg.eig` y sus tres convenciones

Buscar $\lambda$ y $v\ne0$ tales que $Av=\lambda v$. A mano lo hicieron vía
$\det(A-\lambda I)=0$; NumPy **no** resuelve ese polinomio (inviable para $n$ grande):
usa el algoritmo QR iterativo de LAPACK. Tres convenciones, y son la fuente de errores
más común de todo el tema:

1. `np.linalg.eig(A)` devuelve `(valores, P)`.
2. Los vectores propios son las **columnas** de `P`: el $i$-ésimo es `P[:, i]`.
3. Vienen normalizados a **norma 1**, con signo arbitrario.

**Predicción**: la matriz de abajo no es simétrica. ¿Sus valores propios serán
necesariamente reales? ¿Sus vectores propios serán ortogonales entre sí?

In [28]:
A = np.array([[5.0, 4.0, 2.0],
              [0.0, 1.0, 0.0],
              [1.0, 2.0, 4.0]])

valores, P = np.linalg.eig(A)
print("valores propios:", valores)
print("\nP (vectores propios, uno por COLUMNA):\n", P)
print("\nnormas de las columnas:", np.linalg.norm(P, axis=0), " <- todas 1")

# La definición, verificada vector por vector:
print()
for k in range(len(valores)):
    comprobar(f"A·v{k} = λ{k}·v{k}   (λ{k} = {valores[k]:.4f})",
              A @ P[:, k], valores[k] * P[:, k])

# El error clásico, para verlo fallar de una vez (esta FALLA es deliberada):
print()
err_fila = np.max(np.abs(A @ P[0, :] - valores[0] * P[0, :]))
print(f"[FALLA ESPERADA] A·P[0,:] = λ0·P[0,:] usando una FILA           error máx = {err_fila:.2e}")
print("↑ deliberada, y se queda así: demuestra que los vectores propios viven en las COLUMNAS.")
print("\n(Y recuerden: si su P tiene signos opuestos a los del vecino, ambos están bien.)")

valores propios: [6.+0.j 3.+0.j 1.+0.j]

P (vectores propios, uno por COLUMNA):
 [[ 0.8944+0.j -0.7071+0.j -0.5963+0.j]
 [ 0.    +0.j  0.    +0.j  0.7454+0.j]
 [ 0.4472+0.j  0.7071+0.j -0.2981+0.j]]

normas de las columnas: [1. 1. 1.]  <- todas 1

[OK   ] A·v0 = λ0·v0   (λ0 = 6.0000+0.0000j)               error máx = 0.00e+00
[OK   ] A·v1 = λ1·v1   (λ1 = 3.0000+0.0000j)               error máx = 4.44e-16
[OK   ] A·v2 = λ2·v2   (λ2 = 1.0000+0.0000j)               error máx = 5.55e-17

[FALLA ESPERADA] A·P[0,:] = λ0·P[0,:] usando una FILA           error máx = 4.92e+00
↑ deliberada, y se queda así: demuestra que los vectores propios viven en las COLUMNAS.

(Y recuerden: si su P tiene signos opuestos a los del vecino, ambos están bien.)


### 5.2 La forma matricial que vieron en teoría

Las $n$ ecuaciones $Av_i=\lambda_i v_i$, juntas:

$$
AP = PD
\qquad\Longrightarrow\qquad
A = PDP^{-1},\qquad D=\operatorname{diag}(\lambda_1,\dots,\lambda_n),
$$

siempre que $P$ sea invertible (lo cual exige $n$ vectores propios independientes: no
toda matriz es diagonalizable, aquí trabajamos con las que sí). Dos identidades de
regalo que conectan con los bloques anteriores:
$\operatorname{tr}(A)=\sum_i\lambda_i$ y $\det(A)=\prod_i\lambda_i$.

In [29]:
D = np.diag(valores)

comprobar("A P = P D",                    A @ P,                     P @ D)
comprobar("A = P D P⁻¹",                  A,                         P @ D @ np.linalg.inv(P))
comprobar("D = P⁻¹ A P   (diagonaliza)",  np.linalg.inv(P) @ A @ P,  D)
print()
comprobar("tr(A) = Σ λᵢ",  np.trace(A),      np.sum(valores))
comprobar("det(A) = Π λᵢ", np.linalg.det(A), np.prod(valores))

# Como A NO es simétrica, P no es ortogonal (los vectores propios no son perpendiculares):
print("\n¿PᵀP = I?", np.allclose(P.T @ P, np.eye(3)), " -> no; por eso aquí hace falta P⁻¹")

[OK   ] A P = P D                                          error máx = 4.44e-16
[OK   ] A = P D P⁻¹                                        error máx = 8.88e-16
[OK   ] D = P⁻¹ A P   (diagonaliza)                        error máx = 6.23e-16

[OK   ] tr(A) = Σ λᵢ                                       error máx = 0.00e+00
[OK   ] det(A) = Π λᵢ                                      error máx = 3.55e-15

¿PᵀP = I? False  -> no; por eso aquí hace falta P⁻¹


### 5.3 El caso simétrico: `eigh`

Toda matriz de covarianzas, toda Gram $X^\top X$, toda hessiana es **simétrica**, y para
ellas el teorema espectral regala mucho: valores propios **reales**, vectores propios
**ortonormales** ($P^\top P=I$, luego $P^{-1}=P^\top$) y $A=PDP^\top$.
`np.linalg.eigh` explota la simetría: más rápido, más preciso, valores ordenados
ascendentemente, nunca complejos. Con una simétrica, usar `eig` es un error de oficio.

**Predicción**: ¿serán reales estos valores propios? ¿Será $P^\top P=I$?

In [30]:
S = np.array([[ 6.0, -2.0, -1.0],
              [-2.0,  6.0, -1.0],
              [-1.0, -1.0,  5.0]])
print("¿simétrica?", np.allclose(S, S.T))

valS, Ps = np.linalg.eigh(S)
print("valores propios (ordenados):", valS, " <- reales, como prometía la teoría")

comprobar("PᵀP = I (ortonormalidad)", Ps.T @ Ps, np.eye(3))
comprobar("P⁻¹ = Pᵀ (gratis)",        np.linalg.inv(Ps), Ps.T)
comprobar("S = P D Pᵀ",               S, Ps @ np.diag(valS) @ Ps.T)
print("\n¿definida positiva? todos los λ > 0:", np.all(valS > 0))
print("(lectura de optimización: para una función C², la Hessiana es simétrica; si en")
print(" un punto CRÍTICO es definida positiva, ese punto es un mínimo local estricto)")

¿simétrica? True
valores propios (ordenados): [3. 6. 8.]  <- reales, como prometía la teoría
[OK   ] PᵀP = I (ortonormalidad)                           error máx = 4.44e-16
[OK   ] P⁻¹ = Pᵀ (gratis)                                  error máx = 5.55e-16
[OK   ] S = P D Pᵀ                                         error máx = 7.11e-15

¿definida positiva? todos los λ > 0: True
(lectura de optimización: para una función C², la Hessiana es simétrica; si en
 un punto CRÍTICO es definida positiva, ese punto es un mínimo local estricto)


### 5.4 Para qué sirve: la distribución de largo plazo de una economía

Cadena de Markov de **movilidad intergeneracional**: estratos bajo, medio y alto, con
$p_{ij}=\Pr(\text{hijo en } j \mid \text{padre en } i)$, filas que suman 1. La
distribución evoluciona como $\pi_{t+1}=\pi_t P$, y la estacionaria cumple

$$
\pi P = \pi \iff P^\top \pi^\top = \pi^\top :
$$

$\pi$ es el vector propio de $P^\top$ con $\lambda=1$. Toda matriz estocástica tiene
$\lambda_1=1$; la **unicidad** de $\pi$ y la **convergencia** desde cualquier inicio
exigen condiciones adicionales (irreducibilidad y aperiodicidad) — aquí todas las
entradas de $P$ son positivas, lo que las garantiza (Perron–Frobenius). En ese caso,
$|\lambda_2|$ gobierna la **velocidad** de convergencia: cuánto tarda el origen social
en dejar de importar.

Trampa final de normalización: NumPy entrega el vector propio con **norma** 1, pero una
distribución debe **sumar** 1. Se renormaliza a mano.

In [31]:
P_mk = np.array([[0.70, 0.25, 0.05],
                 [0.20, 0.60, 0.20],
                 [0.05, 0.25, 0.70]])
print("filas suman:", P_mk.sum(axis=1), " -> estocástica, y con TODAS las entradas > 0")

w, V = np.linalg.eig(P_mk.T)             # transpuesta: buscamos el vector propio IZQUIERDO
print("valores propios:", np.sort(w)[::-1])

k = np.argmin(np.abs(w - 1.0))
pi = V[:, k]
print(f"\nvector propio crudo (norma 1): {pi}   suma = {pi.sum():.4f}")
pi = pi / pi.sum()                        # de norma 1 a SUMA 1
print(f"π estacionaria:                {pi}   suma = {pi.sum():.4f}")
comprobar("π P = π", pi @ P_mk, pi)

# Verificación independiente: iterar 60 generaciones desde 'todos en estrato bajo'.
pi0 = np.array([1.0, 0.0, 0.0])
comprobar("π₀ P⁶⁰ converge a π", pi0 @ np.linalg.matrix_power(P_mk, 60), pi, atol=1e-6)

lam2 = np.sort(np.abs(w))[-2]
print(f"\n|λ₂| = {lam2:.2f}: el error decae como {lam2:.2f}ᵏ",
      f"-> ~{np.log(0.01)/np.log(lam2):.0f} generaciones para reducirlo 100×.")
print("λ₁ da el destino; |λ₂| da la persistencia. El espectro respondió una pregunta económica.")

filas suman: [1. 1. 1.]  -> estocástica, y con TODAS las entradas > 0
valores propios: [1.  +0.j 0.65+0.j 0.35+0.j]

vector propio crudo (norma 1): [-0.5298+0.j -0.6623+0.j -0.5298+0.j]   suma = -1.7219+0.0000j
π estacionaria:                [0.3077-0.j 0.3846-0.j 0.3077-0.j]   suma = 1.0000+0.0000j
[OK   ] π P = π                                            error máx = 5.55e-17
[OK   ] π₀ P⁶⁰ converge a π                                error máx = 2.98e-12

|λ₂| = 0.65: el error decae como 0.65ᵏ -> ~11 generaciones para reducirlo 100×.
λ₁ da el destino; |λ₂| da la persistencia. El espectro respondió una pregunta económica.


### Comprobamos juntos (2')

Sin ejecutar nada nuevo, respondan: (1) ¿por qué los valores propios de $S$ salieron
reales y los de una matriz cualquiera podrían no serlo? (2) En la Markov, ¿por qué
usamos `eig(P.T)` y no `eig(P)`? (3) Si les entrego un $\pi$ con norma euclidiana 1,
¿qué le falta para ser una distribución?

---
## 6. Puente a la maestría y cierre (10')

En econometría estimarán $y=X\beta+\varepsilon$. La matriz de diseño se construye con
lo aprendido hoy (`np.column_stack`), y el estimador se **escribe**
$\hat\beta=(X^\top X)^{-1}X^\top y$ pero se **calcula** con `np.linalg.lstsq`, que
resuelve el problema de mínimos cuadrados directamente, sin que ustedes formen nunca
$X^\top X$ (cuyo $\kappa$ es el **cuadrado** del de $X$), y que además reporta el rango
y los valores singulares para diagnóstico. Treinta segundos de demostración:

In [32]:
rng = np.random.default_rng(2026)
n = 100
x1 = rng.normal(loc=5.0, scale=2.0, size=n)
y  = 2.0 + 3.0 * x1 + rng.normal(scale=0.5, size=n)     # β verdadero = (2, 3)

X = np.column_stack([np.ones(n), x1])                    # matriz de diseño (n × 2)
beta, residuos, rango, valsing = np.linalg.lstsq(X, y, rcond=None)

print("β estimado:", beta, " <- cerca de (2, 3); la diferencia es el ruido muestral")
print("rango reportado:", rango, "| valores singulares:", valsing)

e = y - X @ beta
comprobar("Xᵀe ≈ 0  (la condición que DEFINE mínimos cuadrados)", X.T @ e,
          np.zeros(2), atol=1e-8)
print("\nEsa ortogonalidad es la puerta de entrada a toda la econometría. Ahí lo dejamos.")

β estimado: [2.179  2.9749]  <- cerca de (2, 3); la diferencia es el ruido muestral
rango reportado: 2 | valores singulares: [55.6085  3.6631]
[OK   ] Xᵀe ≈ 0  (la condición que DEFINE mínimos cuadrados) error máx = 4.17e-12

Esa ortogonalidad es la puerta de entrada a toda la econometría. Ahí lo dejamos.


### Ticket de salida (3 preguntas, en papel o chat)

1. ¿Qué `shape` produce `(3,) @ (3,)`? ¿Y `(1,3) @ (3,1)`? ¿Cuál es un escalar?
2. En una línea: ¿por qué `np.linalg.solve(A, b)` y no `np.linalg.inv(A) @ b`?
3. En la `P` que devuelve `np.linalg.eig`, ¿dónde están los vectores propios (filas o
   columnas) y qué norma tienen?

### Deber (con el cuaderno ampliado)

El cuaderno **ampliado** profundiza cada bloque de hoy y agrega: vistas y copias,
tensores 3D, `slogdet`, matriz de Hilbert y comparación `solve` vs `inv`, valores
propios complejos, potencias $A^k=PD^kP^{-1}$, proyección ortogonal y el desarrollo
completo de mínimos cuadrados con multicolinealidad.

**Deber formativo (no calificado)**: ejercicios **Tú haces 2, 4, 6 y 7** del
ampliado. El **3** (reshape y arreglos 3D) es práctica adicional opcional; el **8**
(multicolinealidad) es un reto avanzado, no obligatorio. Las soluciones colapsadas
quedan disponibles: intenten primero, comparen después.

Entrega: celda Markdown inicial con nombre completo y fecha; en cada ejercicio, código
**más** una interpretación breve de los resultados; antes de entregar, reinicien el
kernel y ejecuten todo de arriba abajo, conservando las salidas. Las comprobaciones de
los ejercicios asignados deben mostrar `[OK]`; la `[FALLA ESPERADA]` del bloque
espectral **se conserva** (demuestra que los vectores propios están en las columnas).
Archivo: `deber_matrices_apellido_nombre.ipynb`. Plazo sugerido: **miércoles 29 de
julio, 23:59**.

### Diccionario mínimo de hoy

| Notación | NumPy |
|---|---|
| $u^\top v$ (1D) | `u @ v` → escalar |
| $A\odot B$ | `A * B` |
| $AB$ | `A @ B` |
| $A^k$ | `np.linalg.matrix_power(A, k)` |
| $\operatorname{tr}(A)$, $\det(A)$ | `np.trace(A)`, `np.linalg.det(A)` |
| $\operatorname{rango}(A)$, $\kappa(A)$ | `np.linalg.matrix_rank(A)`, `np.linalg.cond(A)` |
| $x=A^{-1}b$ | `np.linalg.solve(A, b)` — nunca `inv(A) @ b` |
| $Av=\lambda v$ | `np.linalg.eig(A)`; simétrica → `np.linalg.eigh(A)` |
| $A=PDP^{-1}$ | `valores, P = np.linalg.eig(A)`; `D = np.diag(valores)` |
| $\min\lVert X\beta-y\rVert$ | `np.linalg.lstsq(X, y, rcond=None)` |

### Las siete trampas de hoy

1. `A * B` no es producto matricial; es `A @ B`.
2. `A ** 2` no es $A^2$; es `matrix_power(A, 2)`.
3. Un 1D no es fila ni columna y `v.T` no hace nada.
4. Los vectores propios son **columnas** de `P`, con norma 1 y signo arbitrario.
5. Nunca `==` entre flotantes; `comprobar` / `np.allclose`.
6. Nunca `inv(A) @ b`; `solve(A, b)`.
7. `det == 0` no detecta singularidad; `matrix_rank` sí.